# V10 SegMAN-B

This notebook runs the official SegMAN repository on Linux/Colab. The Windows workstation cannot complete this path reliably because NATTEN has no Windows wheel for the required setup and the selective scan extension requires CUDA compilation.

In [ ]:
# Confirm GPU is connected
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.stdout else 'No GPU found!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
if [ ! -d /content/CV-Assignment2-Group6/.git ]; then
  git clone -b segmentation https://github.com/linenmin/CV-Assignment2-Group6.git /content/CV-Assignment2-Group6
else
  git -C /content/CV-Assignment2-Group6 pull --ff-only
fi

if [ ! -e /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026 ]; then
  ln -s /content/drive/MyDrive/kul-computer-vision-ga-2-2026 /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026
fi

cd '/content/CV-Assignment2-Group6/Semantic segmentation'
pip install -q -e .

In [ ]:
%%bash
set -e
cd '/content/CV-Assignment2-Group6/Semantic segmentation'
if [ ! -d external/SegMAN/.git ]; then
  git clone --depth 1 https://github.com/yunxiangfu2001/SegMAN.git external/SegMAN
fi
mkdir -p external/SegMAN/pretrained
echo 'Put SegMAN_Encoder_b.pth.tar in external/SegMAN/pretrained before training.'
ls -lh external/SegMAN/pretrained || true

Download the official ImageNet-pretrained SegMAN-B encoder from the SegMAN README Google Drive and place it here:

`/content/CV-Assignment2-Group6/Semantic segmentation/external/SegMAN/pretrained/SegMAN_Encoder_b.pth.tar`

Do not continue if that file is missing; random initialization is not a meaningful experiment for this dataset.

In [ ]:
%%bash
set -e
# Official SegMAN dependency stack. This assumes a Linux Colab GPU runtime.
pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
pip install -q -U openmim
mim install -q mmcv-full==1.7.2
pip install -q natten==0.17.3+torch210cu121 -f https://shi-labs.com/natten/wheels/
pip install -q 'numpy<2' 'opencv-python<4.10' 'setuptools<81'
cd '/content/CV-Assignment2-Group6/Semantic segmentation/external/SegMAN'
grep -v '^triton==' requirements.txt > /tmp/segman_requirements_colab.txt
pip install -q -r /tmp/segman_requirements_colab.txt
cd segmentation
pip install -q -v -e .
cd ../kernels/selective_scan
pip install -q .

In [ ]:
%%bash
set -e
cd '/content/CV-Assignment2-Group6/Semantic segmentation'
python scripts/prepare_segman_experiment.py \
  --segman-root ./external/SegMAN \
  --variant b \
  --encoder-checkpoint ./external/SegMAN/pretrained/SegMAN_Encoder_b.pth.tar \
  --batch-size 2 \
  --workers 4

In [ ]:
%%bash
set -e
cd '/content/CV-Assignment2-Group6/Semantic segmentation/external/SegMAN/segmentation'
python tools/train.py local_configs/segman/ga2/segman_b_ga2.py --work-dir outputs/ga2_segman_b

In [ ]:
%%bash
set -e
cd '/content/CV-Assignment2-Group6/Semantic segmentation'
BEST_CKPT=$(ls -1 external/SegMAN/segmentation/outputs/ga2_segman_b/best_mIoU_*.pth | tail -n 1)
echo "BEST_CKPT=${BEST_CKPT}"
python scripts/predict_segman_test.py \
  --config external/SegMAN/segmentation/local_configs/segman/ga2/segman_b_ga2.py \
  --checkpoint "${BEST_CKPT}" \
  --output-dir outputs/predictions/exp_v10_segman_b_test
python scripts/export_submission.py \
  --prediction-dir outputs/predictions/exp_v10_segman_b_test \
  --output-path outputs/submissions/submission_exp_v10_segman_b.csv \
  --classification-fill 0
ls -lh outputs/submissions/submission_exp_v10_segman_b.csv